## Fetch policy

For each organisation number in `companies`, this notebook calls
`https://data.brreg.no/regnskapsregisteret/regnskap/{orgnr}` and writes the
outcome to `financial_data` with `_id` = organisasjonsnummer.

Two things drive the work list, both recomputed on every run:

**Companies with no document at all.** Never attempted, or added to `companies`
by a later register snapshot, or the last attempt returned something
non-definitive — HTTP 500, 429, a network error, or a MongoDB write failure.
These have always retried and still do. They are the ~1,081 persistent HTTP 500
cases plus anything new.

**Companies that answered 404, under the `REFETCH_NO_DATA` setting.** A 404
means "nothing filed as of the day we asked", not "will never file", so these
records go stale — most sharply for AS founded in 2024 or later, which are
precisely the entities about to file for the first time.

| `REFETCH_NO_DATA` | Which 404s are asked again | Requests |
|---|---|---|
| `"never"` | none | original behaviour |
| `"register_signal"` | those where `companies.sisteInnsendteAarsregnskap` is now set | small, targeted |
| `"always"` | all of them | ~726K, hours |

`"register_signal"` is the default. Of the ~726K `no_data` records, the large
majority are ENK and Forening entities that are not required to file at all, and
asking them again on every run is traffic against a public government API that
cannot return anything new. `sisteInnsendteAarsregnskap` is Brreg's own
statement that a filing exists, so intersecting it with the `no_data` set gives
the companies whose 404 has actually been overtaken by events.

Two limits on that signal, both of which belong in the report:

- It only moves when `companies` is reloaded from a newer register snapshot.
  With a stale `companies` collection the targeted re-fetch finds nothing, and
  the coupling between the two ingestion steps is real rather than incidental.
- It inherits the register field's accuracy. `Analyse_data.ipynb` cells 2 and 4
  document disagreement in the other direction — 244 companies where the stored
  API year and `sisteInnsendteAarsregnskap` differ, 113 of them with a reported
  2025 filing and no insolvency flag to explain it. A field that can be wrong
  one way can be wrong the other, so `"always"` is the only setting that
  guarantees a 404 is not simply out of date. Running it occasionally, rather
  than every time, is the reasonable compromise.

### Narrowing by legal form

`LEGAL_FORMS` restricts the whole work list — both never-answered and re-asked
companies — to a set of `organisasjonsform.kode` values. `None` means the whole
register; `{"AS"}` means aksjeselskap only.

The filter is applied to the re-ask set as well as the outstanding set, so a
narrowed run stays narrow rather than pulling the register back in through the
404 side. Two consequences to state if it is used:

- The persistent HTTP 500 cases outside the selected forms stop being retried,
  so `financial_data` coverage for those forms freezes at its current level.
  The shortfall is spread across 16 legal forms, not concentrated in AS.
- Coverage percentages computed against the full register — including the
  analytics layer's left join over all 1,171,373 entities — will show a widening
  gap for the excluded forms that is a consequence of this setting rather than
  of the data. The run summary reports progress register-wide for that reason.

Narrowing to AS is defensible on its own terms: ENK and Forening are near-100%
`no_data` because they are not required to file, so most of the register carries
no financial statements to fetch. It is a scope decision, and the report should
present it as one.

**Successful fetches are never refreshed under any setting.** The API returns
only the latest available filing, so a record fetched on a given date keeps that
filing permanently. Picking up a company's next annual accounts would require
re-requesting the ~445K successes as well, and that is not implemented. The
report should state it.

Re-asking is safe to repeat: results are written with
`replace_one(..., upsert=True)`, so a company that answers 404 again is
overwritten in place with a fresh `fetched_at` rather than duplicated.


In [1]:
import pymongo
import requests
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

# ============================================================
# CONFIGURATION
# ============================================================
BATCH_LIMIT = None        # None = process everything remaining; or set a number to cap this run
MAX_WORKERS = 5            # concurrent requests - keep modest, this is a public gov't API
REQUEST_TIMEOUT = 15       # seconds before giving up on a single request
PROGRESS_EVERY = 100       # print a progress line every N companies processed

# Which already-answered companies to ask again. A 404 means "nothing filed as
# of the day we asked", not "will never file", so it goes stale.
#   "never"           - only companies with no document at all (original behaviour)
#   "register_signal" - plus every no_data company that Brreg now reports a
#                       filing for, via companies.sisteInnsendteAarsregnskap
#   "always"          - plus every no_data company, regardless of the register
REFETCH_NO_DATA = "always"

# Which legal forms to work on, by organisasjonsform.kode.
#   None            - the whole register
#   {"AS"}          - aksjeselskap only (431,581 in the 2026-08-25 snapshot)
#   {"AS", "ASA"}   - add allmennaksjeselskap
# Applies to the whole work list, both never-answered and re-asked companies.
LEGAL_FORMS = {"AS", "ASA"}

BASE_URL = "https://data.brreg.no/regnskapsregisteret/regnskap/"
MONGO_URI = "mongodb://mongodb:27017/"

# ============================================================
# SETUP
# ============================================================
client = pymongo.MongoClient(MONGO_URI)
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# ============================================================
# STEP 1: Figure out what's left to fetch
# ============================================================
print("Loading all organisasjonsnummer from companies collection...")
# sisteInnsendteAarsregnskap rides along on the same cursor: it is Brreg's own
# statement that a filing exists, and it is what makes a targeted re-fetch
# possible without asking the API about 725K entities that never file.
all_org_numbers = set()
selected_orgs = set()
filing_reported = set()
for doc in companies_col.find(
        {}, {"organisasjonsnummer": 1, "sisteInnsendteAarsregnskap": 1,
             "organisasjonsform.kode": 1, "_id": 0}):
    org = doc["organisasjonsnummer"]
    all_org_numbers.add(org)
    if LEGAL_FORMS is None or (doc.get("organisasjonsform") or {}).get("kode") in LEGAL_FORMS:
        selected_orgs.add(org)
    if doc.get("sisteInnsendteAarsregnskap"):
        filing_reported.add(org)
print(f"Total companies: {len(all_org_numbers):,}")
print(f"  Selected forms {sorted(LEGAL_FORMS) if LEGAL_FORMS else 'ALL'}: {len(selected_orgs):,}")
print(f"  Brreg reports a filing for: {len(filing_reported):,}")

print("Loading already-fetched organisasjonsnummer from financial_data collection...")
# One cursor, three uses: the set drives the work list, the counter reports the
# composition, and the no_data ids are the re-fetch candidates.
already_fetched = set()
no_data_ids = set()
status_counts = Counter()
for doc in financial_col.find({}, {"_id": 1, "fetch_status": 1}):
    status = doc.get("fetch_status")
    already_fetched.add(doc["_id"])
    status_counts[status] += 1
    if status == "no_data":
        no_data_ids.add(doc["_id"])
print(f"Already fetched: {len(already_fetched):,}")
for status, n in status_counts.most_common():
    print(f"  {status}: {n:,}")

# Companies with no document at all: never attempted, added by a later register
# snapshot, or last attempt returned a non-definitive response (500, 429,
# network error, Mongo write failure). These retry on every run and always have.
outstanding = sorted(selected_orgs - already_fetched)

# Companies that answered 404 and are worth asking again.
if REFETCH_NO_DATA == "always":
    refetch_pool = no_data_ids
elif REFETCH_NO_DATA == "register_signal":
    refetch_pool = no_data_ids & filing_reported
elif REFETCH_NO_DATA == "never":
    refetch_pool = set()
else:
    raise ValueError(f"REFETCH_NO_DATA must be never/register_signal/always, got {REFETCH_NO_DATA!r}")

# The legal-form filter applies to re-asks as well, so a narrowed run stays
# narrow rather than pulling the whole register back in through the 404 set.
refetch = sorted(refetch_pool & selected_orgs)

# Outstanding first, so a BATCH_LIMIT never starves the genuine retries.
remaining = outstanding + refetch
print(f"\nREFETCH_NO_DATA = {REFETCH_NO_DATA!r}   LEGAL_FORMS = {LEGAL_FORMS!r}")
print(f"Work list: {len(remaining):,}")
print(f"  never answered:      {len(outstanding):,}")
print(f"  no_data to re-ask:   {len(refetch):,}")
print(f"  success, not re-asked: {status_counts['success']:,}")

if BATCH_LIMIT is not None:
    remaining = remaining[:BATCH_LIMIT]
    print(f"BATCH_LIMIT set - processing {len(remaining):,} this run")
else:
    print(f"No BATCH_LIMIT - will process all {len(remaining):,} remaining (stop anytime with the Jupyter Stop button)")
print()

# ============================================================
# STEP 2: Fetch function for a single company
# ============================================================
def fetch_financial_data(org_nr):
    """
    Fetches financial data for one organisasjonsnummer.
    Returns a dict for DEFINITIVE outcomes (success or confirmed no-data).
    Returns None for transient problems, leaving that company unfetched
    so it is automatically retried on a future run.

    A returned dict is written with replace_one(upsert=True), so re-asking a
    company that previously answered 404 overwrites the old record in place,
    including fetched_at. Nothing accumulates.
    """
    url = f"{BASE_URL}{org_nr}"
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
    except requests.exceptions.RequestException as e:
        return None  # network-level failure - transient, retry later

    if response.status_code == 200:
        return {
            "_id": org_nr,
            "organisasjonsnummer": org_nr,
            "fetch_status": "success",
            "http_status": 200,
            "fetched_at": datetime.now(timezone.utc),
            "data": response.json()
        }
    elif response.status_code == 404:
        # Confirmed: no accounts filed for this company - a real, final outcome
        return {
            "_id": org_nr,
            "organisasjonsnummer": org_nr,
            "fetch_status": "no_data",
            "http_status": 404,
            "fetched_at": datetime.now(timezone.utc),
            "data": None
        }
    else:
        # 429 (rate limited) or any other unexpected status - transient, retry later
        return None

def save_result(result):
    """
    Writes one result to MongoDB. Wrapped in try/except so a temporary
    MongoDB outage doesn't crash the whole run - it just skips this
    write, and the company stays eligible for retry next time.
    """
    try:
        financial_col.replace_one({"_id": result["_id"]}, result, upsert=True)
        return True
    except pymongo.errors.PyMongoError as e:
        print(f"  [MONGO WRITE FAILED] {result['_id']}: {e}")
        return False

# ============================================================
# STEP 3: Run, with graceful stop support
# ============================================================
success_count = 0
no_data_count = 0
skipped_count = 0
processed_count = 0
start_time = time.time()

executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
futures = {executor.submit(fetch_financial_data, org_nr): org_nr for org_nr in remaining}

try:
    for future in as_completed(futures):
        result = future.result()
        processed_count += 1

        if result is not None:
            if save_result(result):
                if result["fetch_status"] == "success":
                    success_count += 1
                else:
                    no_data_count += 1
            else:
                skipped_count += 1  # write failed, will retry next run
        else:
            skipped_count += 1  # fetch failed, will retry next run

        if processed_count % PROGRESS_EVERY == 0:
            elapsed = time.time() - start_time
            rate = processed_count / elapsed if elapsed > 0 else 0
            remaining_this_run = len(remaining) - processed_count
            eta_minutes = (remaining_this_run / rate / 60) if rate > 0 else float("inf")
            print(f"[{processed_count:,}/{len(remaining):,}] "
                  f"success={success_count:,} no_data={no_data_count:,} skipped={skipped_count:,} "
                  f"| {rate:.1f} req/s | ETA {eta_minutes:.1f} min")

    executor.shutdown(wait=True)

except KeyboardInterrupt:
    # Triggered by clicking the Jupyter Stop button
    print("\n\n=== STOP REQUESTED ===")
    print("Cancelling queued requests, waiting for in-flight ones to finish...")
    executor.shutdown(wait=True, cancel_futures=True)
    print("Stopped cleanly. All completed work is saved in MongoDB.")
    print("Re-run this cell anytime to continue from where you left off.\n")

# ============================================================
# SUMMARY
# ============================================================
elapsed_total = time.time() - start_time
total_fetched_now = financial_col.count_documents({})
print(f"\n=== RUN SUMMARY ===")
print(f"Processed this run:  {processed_count:,}")
print(f"  Success (200):     {success_count:,}")
print(f"  No data (404):     {no_data_count:,}")
print(f"  Skipped (retry):   {skipped_count:,}")
print(f"Time elapsed:        {elapsed_total/60:.1f} minutes")
print(f"Progress, register-wide: {total_fetched_now:,} / {len(all_org_numbers):,} "
      f"({total_fetched_now/len(all_org_numbers)*100:.1f}%)")

Loading all organisasjonsnummer from companies collection...
Total companies: 1,171,373
  Selected forms ['AS', 'ASA']: 431,796
  Brreg reports a filing for: 449,653
Loading already-fetched organisasjonsnummer from financial_data collection...
Already fetched: 1,170,290
  no_data: 725,646
  success: 444,644

REFETCH_NO_DATA = 'always'   LEGAL_FORMS = {'ASA', 'AS'}
Work list: 27,819
  never answered:      147
  no_data to re-ask:   27,672
  success, not re-asked: 444,644
No BATCH_LIMIT - will process all 27,819 remaining (stop anytime with the Jupyter Stop button)

[100/27,819] success=0 no_data=0 skipped=100 | 29.3 req/s | ETA 15.7 min
[200/27,819] success=6 no_data=47 skipped=147 | 31.8 req/s | ETA 14.5 min
[300/27,819] success=33 no_data=120 skipped=147 | 32.7 req/s | ETA 14.0 min
[400/27,819] success=47 no_data=206 skipped=147 | 33.1 req/s | ETA 13.8 min
[500/27,819] success=55 no_data=298 skipped=147 | 33.2 req/s | ETA 13.7 min
[600/27,819] success=63 no_data=390 skipped=147 | 33.4

In [2]:
import pymongo
import requests
import time
from collections import Counter

BASE_URL = "https://data.brreg.no/regnskapsregisteret/regnskap/"
MONGO_URI = "mongodb://mongodb:27017/"

client = pymongo.MongoClient(MONGO_URI)
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# Recompute the outstanding set exactly as the fetch cell does.
all_org = set(
    d["organisasjonsnummer"]
    for d in companies_col.find({}, {"organisasjonsnummer": 1, "_id": 0})
)
done = set(d["_id"] for d in financial_col.find({}, {"_id": 1}))
remaining = sorted(all_org - done)
print("Remaining: %d\n" % len(remaining))

# Sample sequentially with a deliberate 1-second gap. If these still fail at
# 1 req/s, concurrency is not the cause.
statuses = Counter()
for org in remaining[:20]:
    try:
        r = requests.get(BASE_URL + org, timeout=15)
    except requests.exceptions.RequestException as e:
        statuses["EXCEPTION: %s" % type(e).__name__] += 1
        print("%s  EXCEPTION  %s" % (org, e))
        time.sleep(1.0)
        continue

    statuses[r.status_code] += 1
    retry_after = r.headers.get("Retry-After", "-")
    body = r.text[:150].replace("\n", " ")
    print("%s  HTTP %s  Retry-After=%s  %s" % (org, r.status_code, retry_after, body))
    time.sleep(1.0)

print("\nStatus distribution over the sample:")
for k, v in statuses.most_common():
    print("  %s: %d" % (k, v))

Remaining: 1083

812966022  HTTP 500  Retry-After=-  {"timestamp":"2026-09-11T05:43:30.067+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
813062402  HTTP 500  Retry-After=-  {"timestamp":"2026-09-11T05:43:31.217+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
813918102  HTTP 500  Retry-After=-  {"timestamp":"2026-09-11T05:43:32.368+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
814115232  HTTP 500  Retry-After=-  {"timestamp":"2026-09-11T05:43:33.517+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
815611942  HTTP 500  Retry-After=-  {"timestamp":"2026-09-11T05:43:34.671+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
816521432  HTTP 500  Retry-After=-  {"timestamp"

In [3]:
sample_ids = remaining[:1000]
pipeline = [
    {"$match": {"organisasjonsnummer": {"$in": sample_ids}}},
    {"$group": {"_id": "$organisasjonsform.kode", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
]
for row in companies_col.aggregate(pipeline):
    print("%-8s %d" % (row["_id"], row["n"]))

VPFO     250
STI      200
FLI      147
AS       114
PK       69
SPA      61
NUF      53
ENK      31
DA       26
GFS      23
ASA      16
ANS      5
SA       2
SÆR      1
BA       1
ESEK     1


In [4]:
import random

random.seed(42)
sample = random.sample(remaining, 30)

statuses = Counter()
for org in sample:
    try:
        r = requests.get(BASE_URL + org, timeout=15)
        statuses[r.status_code] += 1
    except requests.exceptions.RequestException as e:
        statuses["EXCEPTION: %s" % type(e).__name__] += 1
    time.sleep(1.0)

print("Random sample of 30 across the full remaining set:")
for k, v in statuses.most_common():
    print("  %s: %d" % (k, v))

# What kind of entities are these? You never posted this from the last cell.
pipeline = [
    {"$match": {"organisasjonsnummer": {"$in": remaining}}},
    {"$group": {"_id": "$organisasjonsform.kode", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
]
print("\nLegal form breakdown of all %d remaining:" % len(remaining))
for row in companies_col.aggregate(pipeline):
    print("  %-8s %d" % (row["_id"], row["n"]))

Random sample of 30 across the full remaining set:
  500: 30

Legal form breakdown of all 1083 remaining:
  VPFO     271
  STI      219
  FLI      159
  AS       129
  PK       71
  SPA      61
  NUF      56
  ENK      33
  DA       32
  GFS      23
  ASA      18
  ANS      5
  SA       3
  SÆR      1
  BA       1
  ESEK     1
